<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-01-logica/04%20-%20Logica%20Proposicional%20Conectivos%20e%20Permissivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos de Partida
## Processo: Planta Industrial de Produção de Biodiesel (Transesterificação em Batelada)

Neste notebook implementamos as funções de avaliação lógica proposicional completas (`AND`, `OR`, `NOT`, `XOR`, `IMPLIES`, `IFF`) e construímos os blocos de permissivos de partida (*Start Permissives*) e intertravamento contínuo (*Run Interlocks / Trips*) para os principais atuadores da planta de biodiesel (Setores 100, 200, 300 e 400).

In [1]:
from typing import Dict, Any
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

Operadores lógicos proposicionais carregados com sucesso.


## 1. Bloco de Permissivo e Trip do Sistema de Aquecimento do Reator (HT-201)

Equações:
$$
P_{\text{HT-201}} \equiv r_1 \land m_{reator} \land l_{reator} \land \neg t_{alta} \land \neg p_1 \land \neg e_1 \land (\text{Auto} \oplus \text{Manual})
$$
$$
\text{Trip}_{\text{HT-201}} \equiv \neg r_1 \lor \neg m_{reator} \lor \neg l_{reator} \lor t_{alta} \lor p_1 \lor e_1
$$

In [2]:
def permissivo_aquecimento_HT201(
    r1: bool,              # Resfriamento de emergência disponível
    m_reator: bool,        # Agitador do reator em operação
    l_reator: bool,        # Nível adequado no reator
    t_alta: bool,          # Alarme de temperatura excessiva
    p1: bool,              # Alarme de sobrepressão
    e1: bool,              # Parada de emergência
    auto_mode: bool,       # Chave seletora Auto
    manual_mode: bool      # Chave seletora Manual
) -> Dict[str, bool]:

    modo_valido = XOR(auto_mode, manual_mode)

    # Condição combinada de permissivo
    permissivo = (
        r1 and
        m_reator and
        l_reator and
        NOT(t_alta) and
        NOT(p1) and
        NOT(e1) and
        modo_valido
    )

    # Condição de trip imediato
    trip = NOT(r1) or NOT(m_reator) or NOT(l_reator) or t_alta or p1 or e1

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Cenários de teste operacional
cenarios_ht201 = [
    {"cenario": "Operação Normal (Auto)", "args": (True, True, True, False, False, False, True, False)},
    {"cenario": "Resfriamento Emergência Indisponível (R1)", "args": (False, True, True, False, False, False, True, False)},
    {"cenario": "Agitador Desligado (Risco Térmico)", "args": (True, False, True, False, False, False, True, False)},
    {"cenario": "Sobretemperatura Reacional", "args": (True, True, True, True, False, False, True, False)},
    {"cenario": "Sobrepressão no Vaso", "args": (True, True, True, False, True, False, True, False)},
    {"cenario": "Emergência Pressionada", "args": (True, True, True, False, False, True, True, False)},
    {"cenario": "Conflito de Modos (Auto & Manual simultâneos)", "args": (True, True, True, False, False, False, True, True)},
]

df_res_ht201 = pd.DataFrame([
    {
        "Cenário": c["cenario"],
        "Permissivo": permissivo_aquecimento_HT201(*c["args"])["Permissivo_Habilitado"],
        "Trip": permissivo_aquecimento_HT201(*c["args"])["Trip_Ativo"],
        "Modo Válido": permissivo_aquecimento_HT201(*c["args"])["Modo_Valido"]
    }
    for c in cenarios_ht201
])
df_res_ht201

,Cenário,Permissivo,Trip,Modo Válido
0,Operação Normal (Auto),True,False,True
1,Resfriamento Emergência Indisponível (R1),False,True,True
2,Agitador Desligado (Risco Térmico),False,True,True
3,Sobretemperatura Reacional,False,True,True
4,Sobrepressão no Vaso,False,True,True
5,Emergência Pressionada,False,True,True
6,Conflito de Modos (Auto & Manual simultâneos),False,False,False


## 2. Bloco de Permissivo da Válvula de Dosagem de Metóxido (XV-202)

Equações:
$$
P_{\text{XV-202}} \equiv m_{reator} \land l_{reator} \land \neg p_1 \land \neg t_{alta} \land \neg l_{alto} \land \neg g_{alm} \land \neg e_1 \land (\text{Auto} \oplus \text{Manual})
$$
$$
\text{Trip}_{\text{XV-202}} \equiv \neg m_{reator} \lor \neg l_{reator} \lor p_1 \lor t_{alta} \lor l_{alto} \lor g_{alm} \lor e_1
$$

In [3]:
def permissivo_valvula_metoxido_XV202(
    m_reator: bool,
    l_reator: bool,
    p1: bool,
    t_alta: bool,
    l_alto: bool,
    g_alm: bool,
    e1: bool,
    auto_mode: bool,
    manual_mode: bool
) -> Dict[str, bool]:
    modo_valido = XOR(auto_mode, manual_mode)
    permissivo = (
        m_reator and
        l_reator and
        NOT(p1) and
        NOT(t_alta) and
        NOT(l_alto) and
        NOT(g_alm) and
        NOT(e1) and
        modo_valido
    )
    trip = NOT(m_reator) or NOT(l_reator) or p1 or t_alta or l_alto or g_alm or e1

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

cenarios_xv202 = [
    {"cenario": "Dosagem Autorizada (Auto)", "args": (True, True, False, False, False, False, False, True, False)},
    {"cenario": "Reator Não Carregado com Óleo", "args": (True, False, False, False, False, False, False, True, False)},
    {"cenario": "Vazamento de Vapores de Metanol (Gás Ativo)", "args": (True, True, False, False, False, True, False, True, False)},
    {"cenario": "Risco de Transbordamento (Nível Alto)", "args": (True, True, False, False, True, False, False, True, False)},
]

pd.DataFrame([
    {
        "Cenário": c["cenario"],
        "Permissivo": permissivo_valvula_metoxido_XV202(*c["args"])["Permissivo_Habilitado"],
        "Trip": permissivo_valvula_metoxido_XV202(*c["args"])["Trip_Ativo"]
    }
    for c in cenarios_xv202
])

,Cenário,Permissivo,Trip
0,Dosagem Autorizada (Auto),True,False
1,Reator Não Carregado com Óleo,False,True
2,Vazamento de Vapores de Metanol (Gás Ativo),False,True
3,Risco de Transbordamento (Nível Alto),False,True


## 3. Demais Blocos Permissivos: Misturador (AG-103), Dreno Glicerina (XV-301) e Bomba Final (P-401)

In [4]:
def permissivo_misturador_metoxido_AG103(l_mix: bool, g_alm: bool, e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    modo_valido = XOR(auto_mode, manual_mode)
    permissivo = l_mix and NOT(g_alm) and NOT(e1) and modo_valido
    trip = NOT(l_mix) or g_alm or e1
    return {'Permissivo': permissivo, 'Trip': trip}

def permissivo_dreno_glicerina_XV301(l_dec: bool, i_glic: bool, e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    modo_valido = XOR(auto_mode, manual_mode)
    permissivo = l_dec and i_glic and NOT(e1) and modo_valido
    trip = NOT(l_dec) or NOT(i_glic) or e1
    return {'Permissivo': permissivo, 'Trip': trip}

def permissivo_bomba_final_P401(f_lav: bool, v_final: bool, l_fim: bool, e1: bool, auto_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    modo_valido = XOR(auto_mode, manual_mode)
    permissivo = f_lav and v_final and NOT(l_fim) and NOT(e1) and modo_valido
    trip = NOT(f_lav) or NOT(v_final) or l_fim or e1
    return {'Permissivo': permissivo, 'Trip': trip}

print("Blocos operacionais AG-103, XV-301 e P-401 definidos.")

Blocos operacionais AG-103, XV-301 e P-401 definidos.


## 4. Geração Automática de Tabela-Verdade para Validação Exaustiva

Avaliação do espaço de estados booleano para o permissivo crítico do sistema de aquecimento $\text{HT-201}$ sob modo automático.

In [5]:
variaveis_ht201 = ['r1', 'm_reator', 'l_reator', 't_alta', 'p1', 'e1']
tabela_ht201 = []

for combo in itertools.product([False, True], repeat=len(variaveis_ht201)):
    st = dict(zip(variaveis_ht201, combo))
    res = permissivo_aquecimento_HT201(
        r1=st['r1'],
        m_reator=st['m_reator'],
        l_reator=st['l_reator'],
        t_alta=st['t_alta'],
        p1=st['p1'],
        e1=st['e1'],
        auto_mode=True,
        manual_mode=False
    )
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela_ht201.append(row)

df_tv_ht201 = pd.DataFrame(tabela_ht201)
print(f"Total de combinações de estados avaliadas: {len(df_tv_ht201)}")
print(f"Combinações seguras que liberam o aquecimento: {df_tv_ht201['Permissivo'].sum()}")
print(f"Combinações que provocam desarme imediato (Trip): {df_tv_ht201['Trip'].sum()}")
print("\nPrimeiras 8 combinações da tabela-verdade:")
print(df_tv_ht201.head(8))
print("\nCombinação segura identificada:")
print(df_tv_ht201[df_tv_ht201['Permissivo'] == True])

Total de combinações de estados avaliadas: 64
Combinações seguras que liberam o aquecimento: 1
Combinações que provocam desarme imediato (Trip): 63

Primeiras 8 combinações da tabela-verdade:
      r1  m_reator  l_reator  t_alta     p1     e1  Permissivo  Trip
0  False     False     False   False  False  False       False  True
1  False     False     False   False  False   True       False  True
2  False     False     False   False   True  False       False  True
3  False     False     False   False   True   True       False  True
4  False     False     False    True  False  False       False  True
5  False     False     False    True  False   True       False  True
6  False     False     False    True   True  False       False  True
7  False     False     False    True   True   True       False  True

Combinação segura identificada:
      r1  m_reator  l_reator  t_alta     p1     e1  Permissivo   Trip
56  True      True      True   False  False  False        True  False
